# 토스 테크니컬 라이팅 가이드 기반 RAG 챗봇

## 이 문서의 유형

이 노트북은 **참조 문서**예요. 토스 테크니컬 라이팅 가이드를 검색 가능한 챗봇으로 만드는
구현 단계와 각 함수의 사용법을 확인하는 용도로 작성했어요.

## 무엇을 만드나요

토스의 [개발자를 위한 글쓰기 기본기](https://github.com/toss/technical-writing) 가이드 23개 문서를
임베딩하고, 질문하면 관련 원칙과 Do/Dont 예시를 찾아 답변하는 RAG 챗봇을 만들어요.

## 진행 순서

1. 가이드 문서를 불러오고 정제해요
2. 문서를 청크 단위로 나누고 벡터 DB에 저장해요
3. 질문과 관련된 문서를 검색해요
4. 검색된 문서를 근거로 답변을 생성해요
5. Gradio로 대화형 화면을 만들어요

> 이 노트북을 작성하면서, 코드의 주석과 함수 이름에도 토스 가이드의
> 문장 원칙(주체를 분명하게, 구체적으로 쓰기, 일관되게 쓰기)을 직접 적용했어요.
> 각 섹션 위에 어떤 원칙을 어디에 적용했는지 표시해 두었어요.


---
## 1. 환경을 설정해요

`.env`에서 API 키를 불러오고, 답변 생성에 사용할 LLM을 만들어요.

`temperature`는 0.1로 낮게 설정했어요. 가이드 원칙을 그대로 인용해서 답변해야 하는
참조 문서 챗봇이라서, 창의적인 답변보다 일관된 답변이 더 중요해요.

**적용 원칙** · 구조 — 가치를 먼저 제공하기: 셀 실행 결과(LLM 객체)를 바로 확인할 수 있게
설정값을 코드 안에 그대로 노출해요.


In [ ]:
from dotenv import load_dotenv
load_dotenv()

from glob import glob
from langchain_openai import ChatOpenAI

# gpt-4.1-mini: 가이드 문서 기반 답변 생성용
# temperature=0.1: 가이드 원칙을 임의로 변형하지 않고 일관되게 인용하기 위해 낮게 설정
llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.1,
    top_p=0.9,
)


---
## 2. 가이드 문서 경로를 모두 찾아요

`technical_writing` 폴더 안의 모든 `.md`, `.mdx` 파일 경로를 가져와요.
이 노트북에는 23개의 가이드 문서가 들어있어요. (`sentence`, `architecture`, `type`, `tutorial` 4개 카테고리)

**적용 원칙** · 구조 — 한 페이지에는 하나만 다루기: "경로 찾기"와 "내용 정제하기"를
하나의 셀에서 처리하지 않고 단계를 분리했어요. 각 단계의 결과를 바로 확인할 수 있어요.


In [ ]:
# 가이드 문서 전체 경로 수집
guide_paths = glob("../data/technical_writing/**/*.mdx", recursive=True)
guide_paths += glob("../data/technical_writing/**/*.md", recursive=True)

print(f"가이드 문서 개수: {len(guide_paths)}개")
for path in guide_paths[:5]:
    print(f"- {path}")


---
## 3. 문서에서 불필요한 부분을 제거해요

원본 `.mdx` 파일에는 RAG 검색에 필요 없는 요소가 섞여 있어요.

| 제거 대상 | 예시 | 제거 이유 |
|---|---|---|
| frontmatter | `--- head: ... ---` | 메타 정보, 본문 내용 아님 |
| import 문 | `import { DoDont } from ...` | 코드 문법, 자연어 아님 |
| 이미지 마크다운 | `![설명](경로)` | 텍스트 검색에 의미 없음 |
| 커스텀 블록 마커 | `::: info`, `:::` | 마커만 제거, 안의 내용은 유지 |
| DoDont 컴포넌트 태그 | `<DoDont>`, `</DoDont.Do>` | 태그만 제거, Do/Dont 예시 텍스트는 유지 |

**적용 원칙** · 문장 — 구체적으로 쓰기: 함수 이름을 `clean_guide_text`로 정하고,
각 정규식 줄에 "무엇을 제거하는지"를 구체적으로 주석으로 남겼어요. "전처리한다" 같은
모호한 표현 대신 제거 대상을 정확히 적었어요.


In [ ]:
import re


def clean_guide_text(raw_text: str) -> str:
    """mdx 원본 텍스트에서 frontmatter, import, 이미지, 컴포넌트 태그를 제거하고
    Do/Dont 예시를 포함한 본문 텍스트만 남긴다.
    """
    text = raw_text

    # frontmatter(--- ~ ---)를 제거한다
    text = re.sub(r"^---.*?---\n", "", text, flags=re.DOTALL)

    # import 문을 제거한다
    text = re.sub(r"^import.*?\n", "", text, flags=re.MULTILINE)

    # 이미지 마크다운(![설명](경로))을 제거한다
    text = re.sub(r"!\[.*?\]\(.*?\)", "", text)

    # 커스텀 블록 마커(::: info 등)를 제거한다. 안의 본문 텍스트는 남긴다
    text = re.sub(r":::\s*\w*\n?", "", text)

    # DoDont 컴포넌트 태그를 제거한다. Do/Dont 예시 텍스트는 남긴다
    text = re.sub(r"</?DoDont[^>]*>", "", text)

    # 빈 줄이 3개 이상 이어지면 2개로 줄인다
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


# 정제 결과를 첫 번째 문서로 확인한다
with open(guide_paths[0], "r", encoding="utf-8") as f:
    sample_raw = f.read()

print(clean_guide_text(sample_raw)[:400])


---
## 4. 문서를 Document 객체로 만들어요

정제한 텍스트에 `category`(가이드 분류), `title`(문서 제목) 메타데이터를 붙여서
LangChain의 `Document` 객체로 만들어요. 이 메타데이터는 나중에 검색 결과를 분류하고
필터링할 때 사용해요.

**적용 원칙** · 문장 — 일관되게 쓰기: 이 노트북에서 만드는 모든 함수는
`동사_명사` 형태로 이름을 짓고, "추출 대상이 무엇인지" 함수 이름에 그대로 드러나게 했어요.
(`extract_category`, `extract_title`) 같은 역할을 하는 함수는 같은 동사로 시작해서
독자가 함수 목록만 보고도 역할을 예측할 수 있게 했어요.


In [ ]:
from langchain_core.documents import Document


def extract_category(path: str) -> str:
    """파일 경로에서 가이드 카테고리(폴더명)를 추출한다.

    예: data/technical_writing/sentence/subject.mdx → "sentence"
    """
    parts = path.replace("\\", "/").split("/")
    if "technical_writing" in parts:
        idx = parts.index("technical_writing")
        return parts[idx + 1]
    return "기타"


def extract_title(content: str) -> str:
    """정제된 본문에서 첫 번째 '# 제목' 줄을 추출한다."""
    match = re.search(r"^#\s+(.+)", content, flags=re.MULTILINE)
    return match.group(1).strip() if match else "제목없음"


def build_guide_documents(paths: list[str]) -> list[Document]:
    """가이드 파일 경로 목록을 받아 Document 객체 목록을 만든다.

    빈 본문(목차 역할만 하는 index.md 등)은 결과에서 제외한다.
    """
    documents = []
    for path in paths:
        with open(path, "r", encoding="utf-8") as f:
            content = clean_guide_text(f.read())

        if not content:
            continue

        documents.append(
            Document(
                page_content=content,
                metadata={
                    "source": path,
                    "category": extract_category(path),
                    "title": extract_title(content),
                },
            )
        )
    return documents


guide_docs = build_guide_documents(guide_paths)

print(f"Document 개수: {len(guide_docs)}개")
print(f"제목: {guide_docs[0].metadata['title']}")
print(f"카테고리: {guide_docs[0].metadata['category']}")


---
## 5. 문서를 검색하기 좋은 크기로 나눠요

가이드 문서 하나에는 보통 "원칙 설명 + 체크리스트 + Do/Dont 예시"가 한 세트로 들어있어요.
청크가 너무 작으면 이 세트가 끊어져서 예시 없이 원칙만 검색되거나, 원칙 없이 예시만
검색될 수 있어요.

`chunk_size=800`은 체크리스트 항목 하나(원칙 + Do/Dont 예시)가 보통 끊기지 않을 정도의
길이예요. 직접 여러 크기로 테스트해 보고 정한 값이에요.

**적용 원칙** · 구조 — 자세히 설명하기: 단순히 "800으로 나눈다"가 아니라,
800을 선택한 이유(체크리스트 한 세트 크기)를 설명에 남겼어요. 처음 보는 사람도 다른
문서에 적용할 때 청크 크기를 어떻게 정해야 할지 판단할 수 있게 했어요.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,       # 체크리스트 항목(원칙+예시) 한 세트가 보통 끊기지 않는 길이
    chunk_overlap=100,    # 항목 경계에서 문맥이 끊기지 않도록 겹치는 구간을 둔다
    length_function=len,
    separators=["\n\n", "\n", " ", ""],
)

guide_chunks = splitter.split_documents(guide_docs)
print(f"청크 개수: {len(guide_chunks)}개")


---
## 6. 청크를 임베딩하고 벡터 DB에 저장해요

한국어 가이드 문서라서 한국어 성능이 좋은 `BAAI/bge-m3` 임베딩 모델을 사용해요.
저장은 로컬 파일 기반인 Chroma를 사용해요. 별도 서버 없이 바로 실행할 수 있고,
이 정도 규모(23개 문서, 70여 개 청크)에는 충분해요.

**적용 원칙** · 구조 — 가치를 먼저 제공하기: 임베딩 모델·벡터 DB 선택 이유를
코드 위쪽 설명에서 먼저 전달하고, 그 다음에 구현 세부사항을 코드로 보여줘요.


In [ ]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_chroma import Chroma
import uuid

embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

guide_vectorstore = Chroma(
    collection_name="toss_technical_writing",
    embedding_function=embedding_model,
    persist_directory="./chroma_guide_db",
)

chunk_ids = [str(uuid.uuid4()) for _ in guide_chunks]
guide_vectorstore.add_documents(guide_chunks, ids=chunk_ids)

print(f"저장된 청크 개수: {len(guide_vectorstore.get()['ids'])}개")


---
## 7. 검색이 잘 되는지 확인해요

검색 방식 2가지를 비교해요.

- **기본 유사도 검색**: 질문과 가장 비슷한 청크 k개를 그대로 반환해요
- **MMR(Maximal Marginal Relevance) 검색**: 비슷한 청크끼리 중복되지 않도록,
  관련성과 다양성을 함께 고려해서 반환해요

가이드처럼 여러 카테고리(문장/구조/유형)에 걸친 질문에는 MMR이 더 다양한 관점의
답변 근거를 가져와요.

**적용 원칙** · 구조 — 예측 가능하게 하기: 검색 결과를 출력하는 형식(`[카테고리] 제목`)을
이후 모든 검색 코드에서 동일하게 사용해서, 어떤 검색 방식을 쓰든 결과를 같은 방식으로
읽을 수 있게 했어요.


In [ ]:
def print_search_results(query: str, docs: list[Document]) -> None:
    """검색 결과를 '[카테고리] 제목' 형식으로 통일해서 출력한다."""
    print(f"질문: {query}")
    for i, doc in enumerate(docs, 1):
        print(f"\n{i}. [{doc.metadata['category']}] {doc.metadata['title']}")
        print(doc.page_content[:150])


# 가이드에 실제로 있는 내용으로 검색을 확인한다
query = "문서에서 주어는 어떻게 써야 하나요?"

basic_results = guide_vectorstore.similarity_search(query, k=3)
print_search_results(query, basic_results)


In [ ]:
# MMR 검색기를 만든다. fetch_k개 후보 중 다양성을 고려해 k개를 고른다
guide_retriever = guide_vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 15, "lambda_mult": 0.5},
)

mmr_results = guide_retriever.invoke(query)
print_search_results(query, mmr_results)


---
## 8. 검색한 문서를 근거로 답변을 생성해요

프롬프트에 다음 4가지 규칙을 명시해요.

1. 가이드 문서에 근거해서 답변한다
2. Do/Dont 예시가 있으면 함께 안내한다
3. 가이드에 없는 내용은 솔직하게 "다루지 않는 내용"이라고 답한다
4. 친절하고 실용적으로 답변한다

**적용 원칙** · 문장 — 주체를 분명하게 하기: 프롬프트의 행동 주체는 항상 '챗봇(당신)'이고,
독자(사용자)가 다음에 무엇을 할 수 있는지를 답변 규칙에 명시했어요.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

guide_prompt = ChatPromptTemplate.from_template("""당신은 토스 테크니컬 라이팅 가이드 전문가입니다.
주어진 가이드 문서를 근거로 질문에 답변하세요.

[가이드 문서]
{context}

[질문]
{question}

[답변 규칙]
1. 가이드 문서에 근거하여 답변하세요.
2. Do/Dont 예시가 있다면 함께 안내하세요.
3. 가이드에 없는 내용은 "이 가이드에서 다루지 않는 내용입니다"라고 답하세요.
4. 친절하고 실용적으로 답변하세요.

[답변]
""")


def format_guide_docs(docs: list[Document]) -> str:
    """검색된 문서들을 '[제목] 본문' 형식으로 이어 붙인다."""
    return "\n\n".join(f"[{doc.metadata['title']}]\n{doc.page_content}" for doc in docs)


guide_rag_chain = (
    {"context": guide_retriever | format_guide_docs, "question": RunnablePassthrough()}
    | guide_prompt
    | llm
    | StrOutputParser()
)

answer = guide_rag_chain.invoke("문서에서 주어는 어떻게 써야 하나요?")
print(answer)


---
## 9. 대화형 챗봇 화면을 만들어요

Gradio `ChatInterface`로 화면을 만들고, 스트리밍으로 답변이 생성되는 과정을 보여줘요.

`examples`에는 이 가이드가 실제로 다루는 질문만 넣었어요. 가이드에 없는 내용(예: UX 버튼
문구)을 예시로 두면, 처음 쓰는 사람이 "이 챗봇이 답을 못하네"라고 오해할 수 있어요.

**적용 원칙** · 구조 — 가치를 먼저 제공하기: 사용자가 화면을 열었을 때 무엇을 물어보면
좋은지 바로 보여줘서, 빈 입력창 앞에서 무엇을 입력할지 고민하는 시간을 줄여요.


In [ ]:
import gradio as gr
from typing import Iterator


def stream_guide_answer(message: str, history: list) -> Iterator[str]:
    """질문을 받아 RAG 체인의 답변을 스트리밍으로 반환한다."""
    partial = ""
    for chunk in guide_rag_chain.stream(message):
        partial += chunk
        yield partial


demo = gr.ChatInterface(
    fn=stream_guide_answer,
    title="토스 테크니컬 라이팅 가이드 챗봇",
    description="토스의 개발자 글쓰기 가이드를 근거로 질문에 답변하는 RAG 챗봇입니다.",
    examples=[
        "문서에서 주어는 어떻게 써야 하나요?",
        "문장을 간결하게 쓰는 방법을 알려주세요",
        "약어는 어떻게 표기해야 하나요?",
        "기술 문서의 제목은 어떻게 정해야 하나요?",
    ],
)

demo.launch()
